# Interactive 1: What does fMRI preprocessing do to the data?

One subject, one 10-minute run, painful heat applied to the lower back (Baliki et al., 2010, *Neuron*).

Each section shows the data **before** and **after** one preprocessing step. Lines marked `# <-- change` are the ones to play with.

In [ ]:
#@title Setup: install packages and download the data (run this first, takes about a minute)
import os, sys
DRIVE_FOLDER = "PASTE_GOOGLE_DRIVE_FOLDER_LINK_HERE"   # shared folder link from Google Drive
SUBJECT = 'cbp001'      # <-- change: 'cbp001', 'cbp006' (chronic back pain) or 'healthy007'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    get_ipython().system('pip install -q nilearn gdown ipyniivue')
    from google.colab import output; output.enable_custom_widget_manager()   # lets the clickable brain viewer render in Colab
    if not os.path.exists('data'):
        get_ipython().system(f'gdown --folder "{DRIVE_FOLDER}" -O data -q')
    DATA = f'data/{SUBJECT}'
else:
    DATA = os.path.join(os.environ.get('FMRI_DATA', 'drive_upload'), SUBJECT)

import numpy as np, nibabel as nib, matplotlib.pyplot as plt, pandas as pd, warnings
from nilearn import plotting, image, masking
from nilearn.datasets import load_mni152_template
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 90
TR = 2.5                                   # seconds between volumes
t = np.arange(240) * TR                    # time axis in seconds
print('files:', sorted(os.listdir(DATA)))

## 1. The raw data: a stack of 3D pictures taken every 2.5 seconds

In [ ]:
raw = nib.load(f'{DATA}/bold_raw.nii.gz')
data = raw.get_fdata()
print('array shape (x, y, z, time):', data.shape)
print('voxel size (mm):', np.round(raw.header.get_zooms()[:3], 2))
print('TR (s):', TR, '   volumes:', data.shape[3], '   run length (min):', data.shape[3] * TR / 60)

plotting.plot_epi(image.index_img(raw, 0), title='volume 0 (the first 3D picture)', cmap='gray', draw_cross=False)
plotting.show()

## 2. A voxel is a time series

In [ ]:
x, y, z = 36, 30, 20      # <-- change: voxel position (x 0-63, y 0-63, z 0-35)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].imshow(data[:, :, z, 0].T, cmap='gray', origin='lower');  ax[0].plot(x, y, 'r+', ms=18, mew=2); ax[0].set_title(f'axial slice z={z}')
ax[1].imshow(data[x, :, :, 0].T, cmap='gray', origin='lower', aspect=3 / 3.44); ax[1].plot(y, z, 'r+', ms=18, mew=2); ax[1].set_title(f'sagittal slice x={x}')
ax[2].plot(t, data[x, y, z, :], color='steelblue'); ax[2].set(xlabel='time (s)', ylabel='MRI signal (arbitrary units)', title='this voxel across the run')
plt.tight_layout(); plt.show()

In [ ]:
# Same thing, but click around: click anywhere in the viewer and the plot underneath shows that voxel's time series
from ipyniivue import NiiVue, SliceType
from ipywidgets import Output
from IPython.display import display, clear_output

image.mean_img(raw).to_filename('mean_raw.nii.gz')
nv = NiiVue(slice_type=SliceType.MULTIPLANAR, height=420)
nv.load_volumes([{'path': 'mean_raw.nii.gz', 'colormap': 'gray'}])
out = Output()

@nv.on_location_change
def show_voxel(loc):
    i, j, k = np.round(np.linalg.inv(raw.affine) @ [*loc['mm'][:3], 1])[:3].astype(int)
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(t, data[i, j, k], color='steelblue'); ax.set(title=f'voxel ({i}, {j}, {k})', xlabel='time (s)')
    plt.close(fig)
    with out:
        clear_output(wait=True); display(fig)

display(nv, out)

## 3. The whole brain across time: does it stay still?

In [ ]:
volumes = [0, 60, 120, 180, 239]      # <-- change: which volumes to look at
z = 20

brain = data.mean(-1) > 0.5 * data.mean()                                  # rough brain mask: voxels brighter than half the average
psc = np.where(brain[..., None], 100 * (data - data.mean(-1, keepdims=True)) / (data.mean(-1, keepdims=True) + 1), np.nan)   # percent signal change, background blanked

fig, ax = plt.subplots(2, len(volumes), figsize=(3.6 * len(volumes), 7))
for i, v in enumerate(volumes):
    ax[0, i].imshow(data[:, :, z, v].T, cmap='gray', origin='lower'); ax[0, i].set_title(f'volume {v}  (t = {v * TR:.0f} s)')
    im = ax[1, i].imshow((psc[:, :, z, v] - psc[:, :, z, 0]).T, cmap='RdBu_r', vmin=-20, vmax=20, origin='lower'); ax[1, i].set_title(f'vol {v} - vol 0  (% change)')
for a in ax.ravel(): a.axis('off')
fig.colorbar(im, ax=ax[1, :], label='% signal change', shrink=0.8); plt.show()

## 4. Head motion correction
Every volume is rigidly shifted and rotated to line up with the first one. The estimated movement is saved as 6 numbers per volume.

In [ ]:
motion = np.loadtxt(f'{DATA}/motion_params.txt')      # columns: 3 rotations (radians), 3 translations (mm)

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].plot(t, np.degrees(motion[:, :3])); ax[0].legend(['pitch', 'roll', 'yaw']); ax[0].set_ylabel('rotation (degrees)')
ax[1].plot(t, motion[:, 3:]);             ax[1].legend(['x', 'y', 'z']);          ax[1].set_ylabel('translation (mm)'); ax[1].set_xlabel('time (s)')
plt.show()

worst = np.abs(motion - motion[0]).sum(1).argmax()
print('volume that moved the most relative to volume 0:', worst, f'(t = {worst * TR:.0f} s)')

In [ ]:
mc = nib.load(f'{DATA}/bold_mc.nii.gz').get_fdata()   # the same run after motion correction
v = worst                                            # <-- change: any volume number

before_diff = np.abs(data[..., v] - data[..., 0]).sum((0, 1)); after_diff = np.abs(mc[..., v] - mc[..., 0]).sum((0, 1))
zbest = 5 + (before_diff - after_diff)[5:30].argmax()                     # slice where motion correction helped the most
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for row, (label, d) in enumerate([('BEFORE motion correction', data), ('AFTER motion correction', mc)]):
    diff = np.where(brain, 100 * (d[..., v] - d[..., 0]) / (d.mean(-1) + 1), np.nan)
    ax[row, 0].imshow(d[:, :, zbest, v].T, cmap='gray', origin='lower');            ax[row, 0].set_title(f'{label}\nvolume {v}, slice z={zbest}')
    ax[row, 1].imshow(diff[:, :, zbest].T, cmap='RdBu_r', vmin=-20, vmax=20, origin='lower'); ax[row, 1].set_title(f'volume {v} minus volume 0 (% signal change)')
    im = ax[row, 2].imshow(diff[32, :, :].T, cmap='RdBu_r', vmin=-20, vmax=20, origin='lower', aspect=3 / 3.44); ax[row, 2].set_title('same difference, sagittal view')
for a in ax.ravel(): a.axis('off')
fig.colorbar(im, ax=ax[:, 2], label='% signal change', shrink=0.6); plt.show()

In [ ]:
# A voxel at the edge of the brain: motion pretends to be brain activity
improvement = np.where(brain, data.std(-1) - mc.std(-1), 0)
x, y, z = np.unravel_index(improvement.argmax(), improvement.shape)   # <-- change: or pick your own x, y, z

fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
ax[0].plot(t, data[x, y, z], color='firebrick');   ax[0].set_title(f'voxel ({x}, {y}, {z}) BEFORE motion correction')
ax[1].plot(t, mc[x, y, z], color='seagreen');      ax[1].set_title('AFTER motion correction')
ax[2].plot(t, np.degrees(motion[:, 0]), color='gray'); ax[2].set_title('estimated head rotation (degrees)'); ax[2].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

## 5. Temporal filtering
Slow drifts (scanner warming up, slow physiology) are removed with a high-pass filter.

In [ ]:
from nilearn.signal import clean
from scipy.signal import welch

HIGH_PASS = 0.01     # Hz  <-- change: try 0.005, 0.02, 0.05
LOW_PASS  = None     # Hz  <-- change: try 0.1 (removes fast noise too)

signal = mc[brain].mean(0)                 # average of all brain voxels = "global signal"
filtered = clean(signal[:, None], t_r=TR, high_pass=HIGH_PASS, low_pass=LOW_PASS, detrend=False, standardize=False)[:, 0]

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].plot(t, signal, color='firebrick', label='before'); ax[0].plot(t, filtered, color='seagreen', label='after'); ax[0].legend(); ax[0].set(xlabel='time (s)', title='global mean signal')
for s, c, l in [(signal, 'firebrick', 'before'), (filtered, 'seagreen', 'after')]:
    f, p = welch(s - s.mean(), fs=1 / TR, nperseg=120); ax[1].semilogy(f, p, color=c, label=l)
ax[1].axvline(HIGH_PASS, ls='--', color='k'); ax[1].legend(); ax[1].set(xlabel='frequency (Hz)', ylabel='power', title='power spectrum')
plt.show()

In [ ]:
x, y, z = 36, 30, 20      # <-- change: a single voxel

def show_voxel_location(x, y, z, ax):
    ax[0].imshow(mc[:, :, z, 0].T, cmap='gray', origin='lower'); ax[0].plot(x, y, 'r+', ms=18, mew=2); ax[0].set_title(f'axial z={z}'); ax[0].axis('off')
    ax[1].imshow(mc[x, :, :, 0].T, cmap='gray', origin='lower', aspect=3 / 3.44); ax[1].plot(y, z, 'r+', ms=18, mew=2); ax[1].set_title(f'sagittal x={x}'); ax[1].axis('off')

signal = mc[x, y, z]
filtered = clean(signal[:, None], t_r=TR, high_pass=HIGH_PASS, low_pass=LOW_PASS, detrend=False, standardize=False)[:, 0]
fig, ax = plt.subplots(1, 3, figsize=(17, 4), width_ratios=[1, 1, 3])
show_voxel_location(x, y, z, ax)
ax[2].plot(t, signal, color='firebrick', label='before'); ax[2].plot(t, filtered, color='seagreen', label='after')
ax[2].legend(); ax[2].set(xlabel='time (s)', title=f'voxel ({x}, {y}, {z})')
plt.tight_layout(); plt.show()

## 6. Spatial smoothing
Each voxel is replaced by a weighted average of its neighbours (Gaussian kernel, width given as FWHM in mm).

In [ ]:
FWHM_LIST = [0, 4, 8, 12]      # <-- change: smoothing kernel sizes in mm

mc_img = nib.load(f'{DATA}/bold_mc.nii.gz')
vol = image.index_img(mc_img, 100)
fig, ax = plt.subplots(1, len(FWHM_LIST), figsize=(4.5 * len(FWHM_LIST), 4.5))
for a, fwhm in zip(ax, FWHM_LIST):
    sm = image.smooth_img(vol, fwhm) if fwhm else vol
    a.imshow(sm.get_fdata()[:, :, 20].T, cmap='gray', origin='lower'); a.set_title(f'FWHM = {fwhm} mm'); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
FWHM = 6                    # <-- change
x, y, z = 36, 30, 20        # <-- change

smoothed = image.smooth_img(mc_img, FWHM).get_fdata()
tsnr_before = mc.mean(-1) / (mc.std(-1) + 1e-6) * brain
tsnr_after = smoothed.mean(-1) / (smoothed.std(-1) + 1e-6) * brain

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].imshow(tsnr_before[:, :, z].T, cmap='hot', vmin=0, vmax=150, origin='lower'); ax[0].set_title('signal-to-noise per voxel, BEFORE smoothing'); ax[0].axis('off')
im = ax[1].imshow(tsnr_after[:, :, z].T,  cmap='hot', vmin=0, vmax=150, origin='lower'); ax[1].set_title(f'AFTER smoothing (FWHM = {FWHM} mm)'); ax[1].axis('off')
fig.colorbar(im, ax=ax, label='temporal SNR (mean / std)', shrink=0.8); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(17, 4), width_ratios=[1, 1, 3])
show_voxel_location(x, y, z, ax)
ax[2].plot(t, mc[x, y, z], color='firebrick', label='before'); ax[2].plot(t, smoothed[x, y, z], color='seagreen', label='after'); ax[2].legend(); ax[2].set(xlabel='time (s)', title=f'voxel ({x}, {y}, {z})')
plt.tight_layout(); plt.show()

## 7. Registration
The functional images are aligned to the subject's own T1 image, and then to a standard template (MNI152), so that a coordinate like (16, 10, -8) means the same place in everyone.

In [ ]:
mni = load_mni152_template(resolution=2)
t1 = nib.load(f'{DATA}/t1_brain.nii.gz')
mean_raw = image.mean_img(raw)
mean_mni = image.mean_img(nib.load(f'{DATA}/bold_mni.nii.gz'))

plotting.plot_anat(mean_raw, title='functional image, scanner space (native)', cmap='gray', draw_cross=False)
plotting.plot_anat(t1, title='T1 anatomical image, scanner space (native)', draw_cross=False)
plotting.plot_anat(mni, title='MNI152 template', draw_cross=False)
plotting.show()

In [ ]:
CUT = (10, 12, -8)      # <-- change: MNI coordinate to look at (this one is the right nucleus accumbens)

d = plotting.plot_anat(mean_raw, cut_coords=CUT, title='BEFORE: native functional image with template outline', cmap='gray'); d.add_edges(mni)
d = plotting.plot_anat(mean_mni, cut_coords=CUT, title='AFTER: registered functional image with template outline', cmap='gray'); d.add_edges(mni)
nac_mask = image.math_img('img > 0', img=f'{DATA}/nac_mask_mni.nii.gz')
plotting.plot_roi(nac_mask, bg_img=mean_mni, cut_coords=CUT, title='nucleus accumbens mask (defined on the template) on the registered functional image')
plotting.show()

## 8. All steps on one voxel

In [ ]:
X_MNI, Y_MNI, Z_MNI = 40, 8, -2      # <-- change: MNI coordinate. (40, 8, -2) is the right anterior insula, a region that responds to pain

from nilearn.maskers import NiftiSpheresMasker
mni_img = nib.load(f'{DATA}/bold_mni.nii.gz')
raw_mni_img = nib.load(f'{DATA}/bold_raw_mni.nii.gz')      # the raw run pushed through the same registration, no motion correction
d = plotting.plot_anat(mean_mni, cut_coords=(X_MNI, Y_MNI, Z_MNI), title='the voxel we are following', cmap='gray')
d.add_markers([(X_MNI, Y_MNI, Z_MNI)], marker_color='red', marker_size=120)
plotting.show()
stim = np.loadtxt(f'{DATA}/stimulus.txt')
sphere = NiftiSpheresMasker([(X_MNI, Y_MNI, Z_MNI)], radius=0)
uncorrected_ts = sphere.fit_transform(raw_mni_img)[:, 0]
raw_ts = sphere.fit_transform(mni_img)[:, 0]
filt_ts = clean(raw_ts[:, None], t_r=TR, high_pass=0.01, detrend=False, standardize=False)[:, 0]
smooth_ts = NiftiSpheresMasker([(X_MNI, Y_MNI, Z_MNI)], radius=0).fit_transform(image.smooth_img(mni_img, 6))[:, 0]
smooth_filt_ts = clean(smooth_ts[:, None], t_r=TR, high_pass=0.01, detrend=False, standardize=False)[:, 0]

fig, ax = plt.subplots(5, 1, figsize=(13, 13), sharex=True)
ax[0].plot(t, uncorrected_ts, color='k');     ax[0].set_title('raw (registered only, no motion correction)')
ax[1].plot(t, raw_ts, color='gray');          ax[1].set_title('motion corrected')
ax[2].plot(t, filt_ts, color='steelblue');    ax[2].set_title('+ high-pass filtered')
ax[3].plot(t, smooth_filt_ts, color='seagreen'); ax[3].set_title('+ smoothed (6 mm)')
ax[4].plot(t, stim, color='firebrick');       ax[4].set_title('heat stimulus (C)'); ax[4].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

In [ ]:
# Same traces on top of each other, as percent signal change. Tick the boxes to choose which ones to show.
from ipywidgets import Checkbox, HBox, interactive_output

traces = {'raw': uncorrected_ts, 'motion corrected': raw_ts, 'high-pass filtered': filt_ts, 'smoothed': smooth_filt_ts}
colors = {'raw': 'k', 'motion corrected': 'gray', 'high-pass filtered': 'steelblue', 'smoothed': 'seagreen'}
boxes = {name: Checkbox(value=(name in ['raw', 'smoothed']), description=name) for name in traces}
boxes['heat stimulus'] = Checkbox(value=True, description='heat stimulus')

def overlay(**show):
    fig, ax = plt.subplots(figsize=(14, 5))
    for name, ts in traces.items():
        if show[name]:
            ax.plot(t, 100 * (ts - ts.mean()) / ts.mean(), color=colors[name], alpha=0.6, label=name)
    ax.set(xlabel='time (s)', ylabel='% signal change', title=f'MNI ({X_MNI}, {Y_MNI}, {Z_MNI})'); ax.legend(loc='upper left')
    if show['heat stimulus']:
        ax2 = ax.twinx(); ax2.plot(t, stim, color='firebrick', alpha=0.3, lw=3); ax2.set_ylabel('temperature (C)', color='firebrick')
    plt.show()

display(HBox(list(boxes.values())), interactive_output(overlay, boxes))